In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_parquet(
    "../data/processed/publication_dataset.parquet"
)

In [3]:
print(df.shape)

df.head()

(3107014, 15)


,id,title,abstract,authors,categories,update_date,published,title_char_count,abstract_char_count,title_word_count,abstract_word_count,author_count,comment_length,doi_exists,version_count
0,704.0001,Calculation of prompt diphoton production cros...,A fully differential calculation in perturba...,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",hep-ph,2008-11-26,1,87,983,12,140,4,39,1,2
1,704.0002,Sparsity-certifying Graph Decompositions,"We describe a new algorithm, the $(k,\ell)$-...",Ileana Streinu and Louis Theran,math.CO cs.CG,2008-12-13,0,40,798,3,115,1,37,0,2
2,704.0003,The evolution of the Earth-Moon system based o...,The evolution of Earth-Moon system is descri...,Hongjun Pan,physics.gen-ph,2008-01-13,0,83,880,14,144,1,19,0,3
3,704.0004,A determinant of Stirling cycle numbers counts...,We show that a determinant of Stirling cycle...,David Callan,math.CO,2007-05-23,0,89,248,11,35,1,8,0,1
4,704.0005,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,In this paper we show how to compute the $\L...,Wael Abu-Shammala and Alberto Torchinsky,math.CA math.FA,2013-10-15,1,52,223,5,37,1,0,0,1


In [4]:
df["text"] = (
    df["title"] +
    " " +
    df["abstract"]
)

In [5]:
df[["title","abstract","text"]].head()

,title,abstract,text
0,Calculation of prompt diphoton production cros...,A fully differential calculation in perturba...,Calculation of prompt diphoton production cros...
1,Sparsity-certifying Graph Decompositions,"We describe a new algorithm, the $(k,\ell)$-...",Sparsity-certifying Graph Decompositions We ...
2,The evolution of the Earth-Moon system based o...,The evolution of Earth-Moon system is descri...,The evolution of the Earth-Moon system based o...
3,A determinant of Stirling cycle numbers counts...,We show that a determinant of Stirling cycle...,A determinant of Stirling cycle numbers counts...
4,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,In this paper we show how to compute the $\L...,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...


In [6]:
df["primary_category"] = (
    df["categories"]
    .str.split()
    .str[0]
)

In [7]:
df[["categories","primary_category"]].head(10)

,categories,primary_category
0,hep-ph,hep-ph
1,math.CO cs.CG,math.CO
2,physics.gen-ph,physics.gen-ph
3,math.CO,math.CO
4,math.CA math.FA,math.CA
5,cond-mat.mes-hall,cond-mat.mes-hall
6,gr-qc,gr-qc
7,cond-mat.mtrl-sci,cond-mat.mtrl-sci
8,astro-ph,astro-ph
9,math.CO,math.CO


In [8]:
from sklearn.model_selection import train_test_split

sample_df, _ = train_test_split(
    df,
    train_size=300000,
    stratify=df["published"],
    random_state=42
)

In [9]:
sample_df.shape

(300000, 17)

In [10]:
sample_df["published"].value_counts(normalize=True)

published
0    0.692547
1    0.307453
Name: proportion, dtype: float64

In [11]:
sample_df = sample_df[
    [
        "text",
        "primary_category",
        "author_count",
        "title_word_count",
        "abstract_word_count",
        "comment_length",
        "doi_exists",
        "version_count",
        "published"
    ]
]

In [12]:
sample_df.shape

(300000, 9)

In [13]:
sample_df.head()

,text,primary_category,author_count,title_word_count,abstract_word_count,comment_length,doi_exists,version_count,published
1074846,Latest results on dark matter searches with H....,astro-ph.HE,2,8,139,41,1,2,1
566132,A Complexity Indicator for 4D Flight Trajector...,math.OC,2,11,93,74,0,3,0
2262910,Meta-Reasoner: Dynamic Guidance for Optimized ...,cs.AI,6,11,193,20,0,6,0
1586598,Adversarial Transformation of Spoofing Attacks...,eess.AS,3,8,193,0,0,1,0
2856711,Asymptotically Optimal Tree-based Group Key Ma...,cs.IT,1,7,113,100,0,1,0


In [14]:
from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

In [15]:
TEXT_FEATURE = "text"

CATEGORY_FEATURE = ["primary_category"]

NUMERIC_FEATURES = [
    "author_count",
    "title_word_count",
    "abstract_word_count",
    "comment_length",
    "doi_exists",
    "version_count"
]

In [16]:
preprocessor = ColumnTransformer(

    transformers=[

        (
            "text",

            TfidfVectorizer(
                max_features=10000,
                stop_words="english",
                ngram_range=(1,2)
            ),

            TEXT_FEATURE
        ),

        (
            "category",

            OneHotEncoder(
                handle_unknown="ignore"
            ),

            CATEGORY_FEATURE
        ),

        (
            "numeric",

            StandardScaler(),

            NUMERIC_FEATURES
        )

    ]

)

In [17]:
pipeline = Pipeline(

    steps=[

        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",

            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )

    ]

)

In [18]:
X = sample_df.drop(
    columns="published"
)

y = sample_df["published"]

In [19]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y

)

In [20]:
pipeline.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['text','primary_category','author_count',...,'comment_length', 'doi_exists','version_count']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('text', ...), ('category', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifyi

In [21]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

y_pred = pipeline.predict(X_test)

y_prob = pipeline.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("ROC AUC :", roc_auc_score(y_test, y_prob))

print(classification_report(y_test, y_pred))

Accuracy : 0.8301166666666666
ROC AUC : 0.8852065376940584
              precision    recall  f1-score   support

           0       0.89      0.86      0.88     41553
           1       0.71      0.75      0.73     18447

    accuracy                           0.83     60000
   macro avg       0.80      0.81      0.80     60000
weighted avg       0.83      0.83      0.83     60000



In [23]:
import time

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

In [24]:
def train_model(model, model_name):

    print("=" * 70)
    print(model_name)
    print("=" * 70)

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", model)
        ]
    )

    start = time.time()

    pipeline.fit(X_train, y_train)

    end = time.time()

    y_pred = pipeline.predict(X_test)

    y_prob = pipeline.predict_proba(X_test)[:, 1]

    print(f"Training Time : {end-start:.2f} sec")
    print(f"Accuracy      : {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision     : {precision_score(y_test, y_pred):.4f}")
    print(f"Recall        : {recall_score(y_test, y_pred):.4f}")
    print(f"F1 Score      : {f1_score(y_test, y_pred):.4f}")
    print(f"ROC AUC       : {roc_auc_score(y_test, y_prob):.4f}")

    print("\nClassification Report\n")

    print(classification_report(y_test, y_pred))

    return pipeline

In [25]:
logistic_model = train_model(

    LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Logistic Regression"
)

Logistic Regression
Training Time : 151.78 sec
Accuracy      : 0.8301
Precision     : 0.7109
Recall        : 0.7541
F1 Score      : 0.7319
ROC AUC       : 0.8852

Classification Report

              precision    recall  f1-score   support

           0       0.89      0.86      0.88     41553
           1       0.71      0.75      0.73     18447

    accuracy                           0.83     60000
   macro avg       0.80      0.81      0.80     60000
weighted avg       0.83      0.83      0.83     60000



In [ ]:
# # Random forest (Random Forest on 10,173 sparse TF-IDF features is not ideal.)

# rf_model = train_model(

#     RandomForestClassifier(

#         n_estimators=100,

#         max_depth=20,

#         n_jobs=-1,

#         random_state=42

#     ),

#     "Random Forest"
# )

In [26]:
xgb_model = train_model(

    XGBClassifier(

        n_estimators=200,

        max_depth=6,

        learning_rate=0.1,

        subsample=0.8,

        colsample_bytree=0.8,

        eval_metric="logloss",

        random_state=42

    ),

    "XGBoost"
)

XGBoost
Training Time : 628.39 sec
Accuracy      : 0.8348
Precision     : 0.7086
Recall        : 0.7859
F1 Score      : 0.7453
ROC AUC       : 0.8910

Classification Report

              precision    recall  f1-score   support

           0       0.90      0.86      0.88     41553
           1       0.71      0.79      0.75     18447

    accuracy                           0.83     60000
   macro avg       0.80      0.82      0.81     60000
weighted avg       0.84      0.83      0.84     60000



What does this tell us?
Logistic Regression

Pros:

Very fast
High accuracy
Easy to interpret
Great baseline

Cons:

Misses more published papers (lower recall)
XGBoost

Pros:

Best overall metrics
Higher recall
Better ROC-AUC
Better F1-score

Cons:

~4× slower to train

In [ ]:
import joblib



joblib.dump(
    logistic_model,
    "../models/publication_prediction_logistic.pkl"
)

['../models/publication_prediction_logistic.pkl']

In [28]:
joblib.dump(
    xgb_model,
    "../models/publication_prediction_xgboost.pkl"
)

['../models/publication_prediction_xgboost.pkl']

In [29]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    logistic_model,
    "../models/publication_prediction_logistic.pkl"
)

joblib.dump(
    xgb_model,
    "../models/publication_prediction_xgboost.pkl"
)

print("Models Saved Successfully.")

Models Saved Successfully.


In [30]:
comparison = pd.DataFrame({

    "Model":[
        "Logistic Regression",
        "XGBoost"
    ],

    "Accuracy":[
        0.8301,
        0.8348
    ],

    "Precision":[
        0.7109,
        0.7086
    ],

    "Recall":[
        0.7541,
        0.7859
    ],

    "F1":[
        0.7319,
        0.7453
    ],

    "ROC_AUC":[
        0.8852,
        0.8910
    ],

    "Training_Time":[
        151.78,
        628.39
    ]

})

comparison

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,Training_Time
0,Logistic Regression,0.8301,0.7109,0.7541,0.7319,0.8852,151.78
1,XGBoost,0.8348,0.7086,0.7859,0.7453,0.8910,628.39


In [31]:
comparison.to_csv(
    "../data/processed/model_comparison.csv",
    index=False
)

In [35]:
model = joblib.load("../models/publication_prediction_xgboost.pkl")

In [36]:
def predict_publication(
    title,
    abstract,
    category,
    author_count,
    comment_length=0,
    doi_exists=0,
    version_count=1
):

    text = title + " " + abstract

    sample = pd.DataFrame({

        "text":[text],

        "primary_category":[category],

        "author_count":[author_count],

        "title_word_count":[len(title.split())],

        "abstract_word_count":[len(abstract.split())],

        "comment_length":[comment_length],

        "doi_exists":[doi_exists],

        "version_count":[version_count]

    })

    prediction = model.predict(sample)[0]

    probability = model.predict_proba(sample)[0][1]

    return prediction, probability

In [37]:
prediction, probability = predict_publication(

    title="Transformer Based Image Classification",

    abstract="""
This paper proposes a transformer-based architecture
for image classification using self-attention mechanisms.
Extensive experiments demonstrate improved performance
over CNN-based baselines.
""",

    category="cs.CV",

    author_count=4,

    comment_length=45,

    doi_exists=0,

    version_count=2
)

print(prediction)
print(probability)

0
0.053684097


In [38]:
def predict_publication(
    model,
    title,
    abstract,
    category,
    author_count,
    comment_length=0,
    doi_exists=0,
    version_count=1
):
    text = title + " " + abstract

    sample = pd.DataFrame({
        "text": [text],
        "primary_category": [category],
        "author_count": [author_count],
        "title_word_count": [len(title.split())],
        "abstract_word_count": [len(abstract.split())],
        "comment_length": [comment_length],
        "doi_exists": [doi_exists],
        "version_count": [version_count]
    })

    prediction = model.predict(sample)[0]
    probability = model.predict_proba(sample)[0][1]

    return prediction, probability

In [39]:
prediction, probability = predict_publication(
    xgb_model,
    title="Transformer Based Image Classification",
    abstract="""This paper proposes a transformer-based architecture
    for image classification using self-attention mechanisms.""",
    category="cs.CV",
    author_count=4,
    comment_length=45,
    doi_exists=0,
    version_count=2
)

In [40]:
print(prediction)
print(probability)

0
0.08543997


In [42]:
published_sample = df[df["published"] == 1].sample(1, random_state=42)

In [ ]:
prediction, probability = predict_publication(
    xgb_model,
    title="Gauged Baryon and Lepton Number in MSSM_4 Bran...",
    abstract="""A recent D-brane model designed to accommoda...""",
    category="cs.CV",
    author_count=4,
    comment_length=45,
    doi_exists=0,
    version_count=2
)
print(prediction)
print(probability)

In [44]:
row = published_sample.iloc[0]

prediction, probability = predict_publication(

    model=xgb_model,

    title=row["title"],

    abstract=row["abstract"],

    category=row["primary_category"],

    author_count=row["author_count"],

    comment_length=row["comment_length"],

    doi_exists=row["doi_exists"],

    version_count=row["version_count"]

)

print("Actual      :", row["published"])
print("Prediction  :", prediction)
print("Probability :", probability)

Actual      : 1
Prediction  : 1
Probability : 0.88375866


In [45]:
import joblib

model = joblib.load("../models/publication_prediction_xgboost.pkl")

prediction, probability = predict_publication(
    model=model,
    title=row["title"],
    abstract=row["abstract"],
    category=row["primary_category"],
    author_count=row["author_count"],
    comment_length=row["comment_length"],
    doi_exists=row["doi_exists"],
    version_count=row["version_count"]
)